# Scale adversarial probing with NVIDIA `garak`

This notebook connects `garak` to a local Python function. The target is deliberately vulnerable; the objective is to learn generator/probe/detector/report flow and then point the same pattern at your own adapter.

In [ ]:
from pathlib import Path
import importlib.util
import json
import shlex
import subprocess
import sys

installed = importlib.util.find_spec("garak") is not None
print("Python:", sys.version.split()[0])
print("garak installed:", installed)

## 1. Discover available plugins in the installed version

Plugin names evolve. Listing probes first is safer than copying an old command from a blog post.

In [ ]:
if installed:
    cmd = [sys.executable, "-m", "garak", "--list_probes"]
    result = subprocess.run(cmd, text=True, capture_output=True, check=False)
    print((result.stdout + "\n" + result.stderr)[:15000])
else:
    print("Install pinned dependencies: pip install -r requirements.txt")

## 2. Scan a Python function target

`vulnerable_target#single` accepts one string and returns one string. Start with the prompt-injection family; narrow to a specific probe after discovery. Keep generations low in a workshop and increase them in scheduled scans.

In [ ]:
scan_cmd = [
    sys.executable, "-m", "garak",
    "--target_type", "function",
    "--target_name", "02_Prompt_Injection_and_Red_Teaming.vulnerable_target#single",
    "--probes", "promptinject",
    "--generations", "1",
]
print("Command:\n", shlex.join(scan_cmd))

if installed:
    # Importing by dotted package path requires the folder to be a package; the
    # direct module form below is reliable when the module folder is on PYTHONPATH.
    env = dict(__import__("os").environ)
    root = str(Path.cwd())
    module_dir = str(Path("02_Prompt_Injection_and_Red_Teaming").resolve())
    env["PYTHONPATH"] = module_dir + __import__("os").pathsep + root + __import__("os").pathsep + env.get("PYTHONPATH", "")
    runnable = scan_cmd.copy()
    runnable[runnable.index("02_Prompt_Injection_and_Red_Teaming.vulnerable_target#single")] = "vulnerable_target#single"
    result = subprocess.run(runnable, text=True, capture_output=True, env=env, check=False)
    print((result.stdout + "\n" + result.stderr)[-20000:])
    print("exit code:", result.returncode)
else:
    print("Skipped: specialist dependency is not installed in this kernel.")

## 3. Production adapter pattern

Create one narrow adapter that turns the scanner’s prompt into your real application request and returns only the assistant text. Capture side effects separately in a sandbox. Never point an unrestricted red-team scan at production tools, customer data, or a live payment/email endpoint.

In [ ]:
adapter_contract = {
    "input": "prompt: str",
    "output": "assistant_text: str",
    "sandbox_requirements": [
        "synthetic tenant and data",
        "non-production credentials",
        "tool side effects disabled or intercepted",
        "request budget and kill switch",
        "raw reports access-controlled",
    ],
    "triage_fields": [
        "probe", "detector", "prompt", "response", "impact",
        "reproducible", "failed_boundary", "regression_test_id"
    ],
}
print(json.dumps(adapter_contract, indent=2))

**Completion evidence:** preserve the garak report, record the exact command and dependency lock, then promote at least one confirmed finding into `02A_attack_harness.ipynb` or your application’s test suite.